# Data Dictionary — Fair Price Finder
**Dataset:** `merged_raw_imputed.csv`  
**Tim:** CC26-PSU164  
**Sumber data:** Fastwork, Sribu, Projects.co.id (hasil scraping + imputation)

---

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "data").exists():
            return candidate
    return current_dir


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data" / "cleaned" / "merged_raw_imputed.csv"

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Jumlah kolom: {df.shape[1]}')
print(f'Jumlah baris: {df.shape[0]}')

Shape: (5461, 10)
Jumlah kolom: 10
Jumlah baris: 5461


## 1. Ringkasan Kolom

In [12]:
summary = pd.DataFrame({
    'Kolom': df.columns,
    'Tipe Data': df.dtypes.values,
    'Non-Null': df.notnull().sum().values,
    'Missing': df.isnull().sum().values,
    'Missing (%)': (df.isnull().sum().values / len(df) * 100).round(2),
    'Unique': df.nunique().values,
})
summary

,Kolom,Tipe Data,Non-Null,Missing,Missing (%),Unique
0,platform,object,5461,0,0.0,3
1,kategori_utama,object,5461,0,0.0,7
2,sub_kategori,object,5461,0,0.0,29
3,judul_listing,object,5461,0,0.0,4891
4,harga,float64,5461,0,0.0,247
5,rating,float64,5461,0,0.0,12
6,jumlah_order,int64,5461,0,0.0,212
7,durasi_hari,int64,5461,0,0.0,63
8,skills,object,5461,0,0.0,268
9,url_listing,object,5461,0,0.0,5101


## 2. Deskripsi Tiap Kolom

---

### `platform`
| | |
|---|---|
| **Tipe** | String (kategorikal) |
| **Deskripsi** | Platform freelance tempat listing diambil |
| **Nilai valid** | `fastwork`, `sribu`, `projects.co.id` |
| **Sumber** | Hasil scraping |
| **Catatan** | projects.co.id adalah marketplace berbasis job posting (client yang bid), berbeda dari fastwork/sribu yang seller-driven |

In [13]:
df['platform'].value_counts().to_frame('jumlah')

,jumlah
platform,
fastwork,4160
sribu,1181
projects.co.id,120


---
### `kategori_utama`
| | |
|---|---|
| **Tipe** | String (kategorikal) |
| **Deskripsi** | Kategori besar layanan freelance |
| **Nilai valid** | 7 kategori (lihat tabel bawah) |
| **Sumber** | Hasil scraping dari struktur kategori platform |

In [14]:
df['kategori_utama'].value_counts().to_frame('jumlah')

,jumlah
kategori_utama,
Web dan Pemrograman,1790
Pemasaran & Periklanan,1223
Grafis & Desain,1008
Penulisan & Penerjemahan,800
Visual & Audio,613
Past Project,25
Lainnya,2


---
### `sub_kategori`
| | |
|---|---|
| **Tipe** | String (kategorikal) |
| **Deskripsi** | Sub-kategori spesifik layanan di dalam kategori utama |
| **Nilai valid** | 29 sub-kategori |
| **Sumber** | Hasil scraping |
| **Catatan** | Fitur terpenting untuk prediksi harga — range harga sangat bervariasi antar sub-kategori |

In [15]:
df.groupby('sub_kategori')['harga'].agg(['count','median']).rename(
    columns={'count':'jumlah','median':'median_harga'}
).sort_values('median_harga', ascending=False).style.format({'median_harga': 'Rp {:,.0f}'})

,jumlah,median_harga
sub_kategori,,
Accounting,19,"Rp 2,750,000"
Others,2,"Rp 1,687,500"
Mobile Application,200,"Rp 1,500,000"
Web Development,400,"Rp 1,400,000"
Data Entry & Mining,25,"Rp 1,250,000"
Machine Learning,192,"Rp 750,000"
Digital Marketing,400,"Rp 650,000"
All,25,"Rp 500,000"
Social Media Mgmt,200,"Rp 500,000"


---
### `judul_listing`
| | |
|---|---|
| **Tipe** | String (teks bebas) |
| **Deskripsi** | Judul layanan yang ditampilkan oleh freelancer di platform |
| **Contoh** | `"Pembuatan Website Custom Professional (React, Next js, Laravel)"` |
| **Sumber** | Hasil scraping |
| **Catatan** | Digunakan untuk inferensi skills saat imputation. Tidak dipakai langsung sebagai fitur model |

---
### `harga`
| | |
|---|---|
| **Tipe** | Float (numerik kontinu) |
| **Deskripsi** | Harga listing dalam Rupiah (IDR) |
| **Range valid** | Rp 40.000 – Rp 50.000.000 |
| **Satuan** | IDR (Rupiah) |
| **Sumber** | Hasil scraping; 10 missing diisi dengan median per platform + sub_kategori |
| **Catatan** | **Target variable** untuk model prediksi. Distribusi sangat right-skewed → perlu log-transform sebelum training |

In [16]:
df['harga'].describe().apply(lambda x: f'Rp {x:,.0f}')

count         Rp 5,461
mean        Rp 691,457
std       Rp 1,720,849
min          Rp 40,000
25%         Rp 150,000
50%         Rp 300,000
75%         Rp 650,000
max      Rp 50,000,000
Name: harga, dtype: object

---
### `rating`
| | |
|---|---|
| **Tipe** | Float (numerik diskrit) |
| **Deskripsi** | Rating rata-rata yang diberikan klien kepada freelancer |
| **Range valid** | 0.0 – 5.0 |
| **Nilai khusus** | `0.0` = seller baru / belum pernah mendapat order |
| **Sumber** | Hasil scraping; missing diisi: seller baru → 0.0, projects.co.id → median global |
| **Catatan** | ~40% listing punya rating 0 (seller baru). Disarankan buat flag `is_new_seller` untuk modeling |

In [17]:
df['rating'].value_counts().sort_index().to_frame('jumlah')

,jumlah
rating,
0.0,2284
4.0,10
4.1,1
4.2,1
4.3,4
4.4,2
4.5,26
4.6,34
4.7,99


---
### `jumlah_order`
| | |
|---|---|
| **Tipe** | Integer (numerik diskrit) |
| **Deskripsi** | Total order yang pernah diselesaikan freelancer untuk listing ini |
| **Range valid** | 0 – 50.447 |
| **Nilai khusus** | `0` = seller baru / belum ada order |
| **Sumber** | Hasil scraping; missing diisi berdasarkan status rating (0 jika baru, median jika aktif) |
| **Catatan** | Terdapat 1 outlier ekstrem (50.447 order) — kemungkinan top seller Fastwork. Pertimbangkan log-transform atau clip |

In [18]:
df['jumlah_order'].describe().apply(lambda x: f'{x:,.0f}')

count     5,461
mean         22
std         685
min           0
25%           0
50%           1
75%           4
max      50,447
Name: jumlah_order, dtype: object

---
### `durasi_hari`
| | |
|---|---|
| **Tipe** | Integer (numerik diskrit) |
| **Deskripsi** | Estimasi durasi pengerjaan dalam hari yang dijanjikan freelancer |
| **Range valid** | 1 – 30 hari (wajar); hingga 9786 hari di data (anomali) |
| **Satuan** | Hari |
| **Sumber** | Hasil scraping; 726 missing diisi dengan median per platform + sub_kategori |
| **Catatan** | Durasi > 365 hari (22 baris) kemungkinan data entry error — pertimbangkan di-clip ke 365 |

In [19]:
df['durasi_hari'].describe().apply(lambda x: f'{x:,.0f}')

count    5,461
mean        44
std        615
min          0
25%          1
50%          2
75%          5
max      9,786
Name: durasi_hari, dtype: object

---
### `skills`
| | |
|---|---|
| **Tipe** | String (multi-value, dipisah koma) |
| **Deskripsi** | Daftar skill/teknologi yang dikuasai freelancer untuk listing ini |
| **Contoh** | `"React, Laravel, WordPress"` |
| **Sumber** | Hasil scraping; 4218 missing (77%) diisi dengan inferensi dari sub_kategori + judul_listing |
| **Catatan** | Perlu multi-label encoding sebelum masuk model. Jumlah unique skill: 268 kombinasi |

In [20]:
from collections import Counter
all_skills = []
for s in df['skills'].dropna():
    all_skills.extend([x.strip() for x in str(s).split(',')])
top10 = Counter(all_skills).most_common(10)
pd.DataFrame(top10, columns=['skill', 'jumlah'])

,skill,jumlah
0,Python,718
1,Adobe Illustrator,583
2,Adobe Premiere,580
3,Copywriting,516
4,SEO,452
5,Terjemahan,399
6,Bahasa Inggris,399
7,Bahasa Indonesia,399
8,Video Editing,399
9,Canva,398


---
### `url_listing`
| | |
|---|---|
| **Tipe** | String (URL) |
| **Deskripsi** | URL langsung ke halaman listing di platform |
| **Contoh** | `https://fastwork.id/user/xxx/web-development-...` |
| **Sumber** | Hasil scraping |
| **Catatan** | Tidak digunakan sebagai fitur model. Berguna untuk verifikasi data manual |

---
## 3. Catatan Imputation

| Kolom | Jumlah Missing | Metode Imputation | Alasan |
|---|---|---|---|
| `harga` | 10 | Median per platform + sub_kategori | Harga sangat dipengaruhi jenis layanan dan platform |
| `durasi_hari` | 726 | Median per platform + sub_kategori | Durasi bervariasi per jenis pekerjaan |
| `rating` | 2405 | Seller baru → 0.0; ada order → median platform; projects.co.id → median global | Seller baru memang belum punya rating |
| `jumlah_order` | 3300 | Seller baru → 0; aktif → median per platform + sub_kategori | Konsisten dengan logika rating |
| `skills` | 4218 | Inferensi dari sub_kategori + keyword di judul_listing | 77% missing; tidak ada nilai lain yang bisa dipakai |

## 4. Rekomendasi untuk Feature Engineering

| Fitur Baru | Derivasi dari | Tujuan |
|---|---|---|
| `log_harga` | log1p(harga) | Normalisasi target variable |
| `log_order` | log1p(jumlah_order) | Normalisasi distribusi skewed |
| `is_new_seller` | rating == 0 | Flag seller baru sebagai kategori tersendiri |
| `level` | kombinasi rating + jumlah_order | Segmentasi baru/menengah/senior |
| `durasi_hari_clip` | clip(durasi_hari, 0, 365) | Hapus anomali durasi ekstrem |